In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path
import vectorbt as vbt
import sys

sys.path.insert(0, str(Path('.').resolve().parent))
from dema_strat import load_ohlcv, vbt_frame, free
from constants import CASH, COMMISSION, SESSIONS

TABLE_NAME = "f1_session"
SMOOTH_PERIOD = 12

def load_data(file_path, *, verbose=False, use_vwap=False):
    return load_ohlcv(file_path, use_cache=True, add_vwap=use_vwap, verbose=verbose)

def session_mask(index, session):
    start = pd.Timestamp(session["start"]).time()
    end = pd.Timestamp(session["end"]).time()
    tod = np.array(index.to_series().dt.time.values)
    if start <= end:
        return (tod >= start) & (tod < end)
    return (tod >= start) | (tod < end)

def make_signals(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD):
    close = data["Close"]
    high = data["High"]
    low = data["Low"]

    ema1 = vbt.indicators.MA.run(close, window=n1, ewm=True)
    smoothed = vbt.indicators.MA.run(ema1.ma, window=smooth_period, ewm=True)
    atr = vbt.indicators.ATR.run(high, low, close, window=atr_period)

    sl_dist = sl_mult * atr.atr
    tp_dist = rr_ratio * sl_dist
    sl_pct = sl_dist / close
    tp_pct = tp_dist / close

    long_mask = (smoothed.ma > ema1.ma) & np.isfinite(atr.atr) & (atr.atr > 0)
    short_mask = (ema1.ma > smoothed.ma) & np.isfinite(atr.atr) & (atr.atr > 0)

    if use_vwap:
        long_mask = long_mask & (close > data["VWAP"])
        short_mask = short_mask & (close < data["VWAP"])

    return pd.DataFrame({
        "ema1": ema1.ma,
        "smoothed": smoothed.ma,
        "atr": atr.atr,
        "sl_pct": sl_pct,
        "tp_pct": tp_pct,
        "long": long_mask,
        "short": short_mask,
    })

def run_backtest(data, n1=9, atr_period=14, sl_mult=1.0, rr_ratio=0.5, use_vwap=False, smooth_period=SMOOTH_PERIOD, session=None):
    signals = make_signals(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period)
    freq = data.index.to_series().diff().median()

    long_signal = signals["long"].to_numpy(dtype=bool)
    short_signal = signals["short"].to_numpy(dtype=bool)

    n = len(data)
    in_session = np.ones(n, dtype=bool)
    session_end = np.zeros(n, dtype=bool)
    if session is not None:
        in_session = session_mask(data.index, session)
        next_in = np.empty(n, dtype=bool)
        next_in[:-1] = in_session[1:]
        next_in[-1] = False
        session_end = in_session & ~next_in

    long_signal = long_signal & in_session & ~session_end
    short_signal = short_signal & in_session & ~session_end
    exits_long = short_signal | session_end
    exits_short = long_signal | session_end

    close_np = data["Close"].to_numpy(dtype=np.float64)

    portfolio = vbt.Portfolio.from_signals(
        close=close_np,
        entries=long_signal,
        exits=exits_long,
        short_entries=short_signal,
        short_exits=exits_short,
        init_cash=CASH,
        fees=COMMISSION,
        size=100.0,
        size_type="value",
        freq=freq,
        sl_stop=signals["sl_pct"].to_numpy(dtype=np.float64),
        tp_stop=signals["tp_pct"].to_numpy(dtype=np.float64),
    )

    return portfolio, portfolio.stats(), signals

def run_params(data, n1, atr_period, sl_mult, rr_ratio, use_vwap=False, asset=None, timeframe=None, smooth_period=SMOOTH_PERIOD, session=None):
    session_name = session["name"] if session else None
    try:
        portfolio, stats, _ = run_backtest(data, n1=n1, atr_period=atr_period, sl_mult=sl_mult, rr_ratio=rr_ratio, use_vwap=use_vwap, smooth_period=smooth_period, session=session)
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": float(stats["Total Return [%]"]) if stats.get("Total Return [%]") is not None else None,
            "max_dd": float(stats["Max Drawdown [%]"]) if stats.get("Max Drawdown [%]") is not None else None,
            "sharpe": float(stats["Sharpe Ratio"]) if stats.get("Sharpe Ratio") is not None else None,
            "win_rate": float(stats["Win Rate [%]"]) if stats.get("Win Rate [%]") is not None else None,
            "profit_factor": float(stats["Profit Factor"]) if stats.get("Profit Factor") is not None else None,
            "expectancy": float(stats["Expectancy"]) if stats.get("Expectancy") is not None else None,
            "trades": int(stats["Total Trades"]) if stats.get("Total Trades") is not None else None,
        }
    except Exception as e:
        return {
            "n1": n1, "atr_period": atr_period,
            "sl_mult": sl_mult, "rr_ratio": rr_ratio,
            "use_vwap": use_vwap,
            "commission": COMMISSION,
            "smooth_period": smooth_period,
            "asset": asset,
            "timeframe": timeframe,
            "session": session_name,
            "return_pct": None, "max_dd": None, "sharpe": None,
            "win_rate": None, "profit_factor": None,
            "expectancy": None, "trades": None,
        }


In [2]:
file_path = Path("~/market_data/XAUUSD/XAUUSD_M15.csv").expanduser()
data = load_data(file_path, verbose=True)

portfolio, stats, signals = run_backtest(
    data,
    n1=5,
    rr_ratio=2,
    atr_period=18,
    session=SESSIONS[1],
    sl_mult=3,
    use_vwap=1,
    smooth_period=12,
)
print(stats)

trades = portfolio.trades.records_readable
print(trades)

free(data, signals)

[loader] cache hit: XAUUSD_M15__cache_v1.parquet
Start                                                 0
End                                              139917
Period                               1457 days 11:30:00
Start Value                                     50000.0
End Value                                  50007.464764
Total Return [%]                                0.01493
Benchmark Return [%]                         184.221285
Max Gross Exposure [%]                         0.203878
Total Fees Paid                               56.402929
Max Drawdown [%]                               0.030839
Max Drawdown Duration                 724 days 06:45:00
Total Trades                                       1880
Total Closed Trades                                1880
Total Open Trades                                     0
Open Trade PnL                                      0.0
Win Rate [%]                                   41.43617
Best Trade [%]                                 5.487051

In [4]:
# portfolio.plot()
portfolio.value().vbt.plot()

FigureWidget({
    'data': [{'showlegend': False,
              'type': 'scatter',
              'uid': '122b6951-d8cb-490c-ab47-8e56bb7a2962',
              'x': {'bdata': ('AAAAAAEAAAACAAAAAwAAAAQAAAAFAA' ... 'CJIgIAiiICAIsiAgCMIgIAjSICAA=='),
                    'dtype': 'i4'},
              'y': {'bdata': ('AAAAAABq6EAAAAAAAGroQAAAAAAAau' ... 'jf7mroQP0BWN/uauhA/QFY3+5q6EA='),
                    'dtype': 'f8'}}],
    'layout': {'height': 350,
               'legend': {'orientation': 'h',
                          'traceorder': 'normal',
                          'x': 1,
                          'xanchor': 'right',
                          'y': 1.02,
                          'yanchor': 'bottom'},
               'margin': {'b': 30, 'l': 30, 'r': 30, 't': 30},
               'template': '...',
               'width': 700}
})

## Optimization

In [5]:

from itertools import product
import time


db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute(f"DROP TABLE IF EXISTS {TABLE_NAME}")
cursor.execute(f"""
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    n1 INTEGER NOT NULL,
    atr_period INTEGER NOT NULL,
    sl_mult REAL NOT NULL,
    rr_ratio REAL NOT NULL,
    use_vwap INTEGER NOT NULL,
    commission REAL NOT NULL,
    smooth_period INTEGER NOT NULL,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    session TEXT,
    expectancy REAL,
    win_rate REAL,
    max_dd REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    num_trades INTEGER,
    return_pct REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")
conn.commit()

grid = list(product(
    ["M1", "M5", "M15"], # timeframes
    range(5, 15, 2), # fast ema,
    range(6, 16, 2), # smoothing
    [14, 18, 21], # atr period
    [1.0, 2.0, 3.0], # sl mult
    [0.25, 0.5, 1.0, 1.5, 2.0], # rr ratio
    [False, True], # use vwap
    SESSIONS, # session windows (closes positions at end)
))

#* Only for test
# grid = list(product(
#     ["M1"], # timeframes
#     [9], # fast ema
#     [12], # smoothing
#     [14], # atr period
#     [1], # sl mult
#     [1], # rr ratio
#     [True], # use vwap
#     SESSIONS, # sessions
# ))

t0 = time.time()

print(f"[{time.strftime('%H:%M:%S')}] Testing {len(grid)} combinations -> {TABLE_NAME}")
asset = "XAUUSD"
data_by_tf: dict[tuple[str, bool], pd.DataFrame] = {}
for tf in sorted({g[0] for g in grid}):
    for uv in sorted({g[6] for g in grid if g[0] == tf}):
        fp = Path(f"~/market_data/{asset}/{asset}_{tf}.csv").expanduser()
        data_by_tf[(tf, uv)] = load_data(fp, verbose=True, use_vwap=uv)

ONE_LOOP_MEAN_TIME = 0.0
for i, (timeframe, n1, smoothing, ap, sm, rr, uv, session) in enumerate(grid):

    data = data_by_tf[(timeframe, uv)]

    t1 = time.time()
    result = run_params(data, n1, ap, sm, rr, use_vwap=uv, asset=asset, timeframe=timeframe, smooth_period=smoothing, session=session)
    elapsed = time.time() - t1
    cursor.execute(
        f"INSERT INTO {TABLE_NAME} (n1, atr_period, sl_mult, rr_ratio, use_vwap, commission, smooth_period, asset, timeframe, session, expectancy, win_rate, max_dd, sharpe_ratio, profit_factor, num_trades, return_pct) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (result["n1"], result["atr_period"], result["sl_mult"], result["rr_ratio"], uv, result["commission"], result["smooth_period"], result["asset"], result["timeframe"], result["session"],
         result["expectancy"], result["win_rate"], result["max_dd"], result["sharpe"], result["profit_factor"], result["trades"], result["return_pct"]),
    )
    conn.commit()
    ONE_LOOP_MEAN_TIME = (elapsed + ONE_LOOP_MEAN_TIME * i) / (i + 1) if i else elapsed
    combinations_left = len(grid) - (i + 1)
    est_left_sec = ONE_LOOP_MEAN_TIME * combinations_left
    if (i + 1) % max(1, len(grid) // 100) == 0 or (i + 1) == len(grid):
        print(
            f"[{time.strftime('%H:%M:%S')}] {i+1:>5}/{len(grid)} tf={timeframe} n1={n1} ap={ap} sm={sm} rr={rr} vwap={uv} sess={session['name']} "
            f"| sharpe={result['sharpe']} ret={result['return_pct']} trades={result['trades']} ({elapsed:.1f}s) "
            f"| mean={ONE_LOOP_MEAN_TIME:.1f}s est_left={est_left_sec/60:.1f}min"
        )

    free()

free(data_by_tf)

sweep_df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
sweep_df = sweep_df.dropna(subset=["expectancy"])
sweep_df = sweep_df.sort_values("expectancy", ascending=False).reset_index(drop=True)
print(f"Total grid: {len(grid)} | Non-null: {len(sweep_df)}")
sweep_df.head(20)



[11:09:44] Testing 27000 combinations -> f1_session
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M1__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M15__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[loader] cache hit: XAUUSD_M5__cache_v1.parquet
[11:15:50]   270/27000 tf=M1 n1=5 ap=21 sm=1.0 rr=1.5 vwap=True sess=LONDON | sharpe=-38.31233899632021 ret=-3.2812175139840765 trades=57123 (1.1s) | mean=1.2s est_left=519.5min
[11:21:38]   540/27000 tf=M1 n1=5 ap=18 sm=2.0 rr=1.0 vwap=False sess=OVERLAP | sharpe=-23.763482109139293 ret=-1.621369891909689 trades=26706 (1.1s) | mean=1.1s est_left=502.9min
[11:27:24]   810/27000 tf=M1 n1=5 ap=14 sm=3.0 rr=0.5 vwap=False sess=LONDON | sharpe=-44.01884141063211 ret=-5.014946136897466 trades=88765 (1.1s) | mean=1.1s est_left=493.2min
[11:33:10]  1080/27000 tf=M1 n1=5 ap=21 sm=3.0 rr=2.0 vwap=True sess=OVERLAP | sharpe=-2.551077141755734 ret=-0.1575246106

,id,n1,atr_period,sl_mult,rr_ratio,use_vwap,commission,smooth_period,asset,timeframe,session,expectancy,win_rate,max_dd,sharpe_ratio,profit_factor,num_trades,return_pct,run_timestamp
0,19318,5,18,3.0,2.0,1,0.00015,12,XAUUSD,M15,LONDON,0.003971,41.436170,0.030839,0.177943,1.022929,1880,0.014930,2026-08-24 08:48:15
1,19198,5,14,3.0,2.0,1,0.00015,12,XAUUSD,M15,LONDON,0.003828,41.511442,0.033459,0.170933,1.022044,1879,0.014385,2026-08-24 08:47:28
2,19438,5,21,3.0,2.0,1,0.00015,12,XAUUSD,M15,LONDON,0.003543,41.392136,0.030883,0.159218,1.020434,1882,0.013336,2026-08-24 08:49:01
3,19310,5,18,3.0,1.5,1,0.00015,12,XAUUSD,M15,LONDON,0.002440,42.160461,0.033368,0.114216,1.014291,1907,0.009305,2026-08-24 08:48:11
4,20398,7,18,3.0,2.0,1,0.00015,8,XAUUSD,M15,LONDON,0.002264,40.717949,0.035826,0.102869,1.012819,1950,0.008828,2026-08-24 08:54:51
5,18958,5,18,3.0,2.0,1,0.00015,10,XAUUSD,M15,LONDON,0.002245,41.692955,0.035502,0.099308,1.012369,1902,0.008539,2026-08-24 08:45:43
6,21838,9,18,3.0,2.0,1,0.00015,6,XAUUSD,M15,LONDON,0.002104,40.927835,0.037212,0.095095,1.011832,1940,0.008162,2026-08-24 09:02:36
7,19190,5,14,3.0,1.5,1,0.00015,12,XAUUSD,M15,LONDON,0.001983,42.257218,0.033220,0.092447,1.011583,1905,0.007555,2026-08-24 08:47:25
8,19182,5,14,3.0,1.0,1,0.00015,12,XAUUSD,M15,LONDON,0.001968,45.856634,0.030239,0.099866,1.012137,1967,0.007743,2026-08-24 08:47:22
9,19078,5,21,3.0,2.0,1,0.00015,10,XAUUSD,M15,LONDON,0.001662,41.605456,0.034234,0.073802,1.009133,1906,0.006335,2026-08-24 08:46:41


In [2]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)

vwap_comparison = pd.read_sql_query(
    f"""SELECT use_vwap,
              COUNT(*) AS n,
              AVG(expectancy) AS mean_expectancy,
              AVG(win_rate) AS mean_win_rate,
              AVG(max_dd) AS mean_max_dd,
              AVG(sharpe_ratio) AS mean_sharpe,
              AVG(profit_factor) AS mean_profit_factor,
              AVG(num_trades) AS mean_num_trades,
              AVG(return_pct) AS mean_return_pct
       FROM {TABLE_NAME}
       WHERE sharpe_ratio IS NOT NULL
       GROUP BY use_vwap""",
    conn,
)
print(vwap_comparison)


   use_vwap      n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0         0  13500        -0.031695      47.737581     1.976691   -23.942296   
1         1  13500        -0.024590      43.924844     0.640468   -11.574413   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            0.528398     33187.526148        -1.974780  
1            0.642986     11397.682000        -0.636665  


In [4]:
n1_comparison = pd.read_sql_query(
    """
    SELECT n1, smooth_period,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE sharpe_ratio IS NOT NULL
    GROUP BY n1, smooth_period
    ORDER BY mean_expectancy  
    """,
    conn,
)
print(n1_comparison)

    n1  smooth_period     n  mean_expectancy  mean_win_rate  mean_max_dd  \
0   13             12  1080        -0.029405      45.354962     1.206687   
1   11             14  1080        -0.029334      45.380018     1.207716   
2   11             12  1080        -0.029074      45.495820     1.223324   
3   13             10  1080        -0.029025      45.537288     1.224711   
4    9             14  1080        -0.028973      45.611957     1.227601   
5   13             14  1080        -0.028508      45.398653     1.184324   
6    5              6  1080        -0.028248      45.524061     1.555670   
7    7              6  1080        -0.028231      45.747506     1.455423   
8    9              8  1080        -0.028183      45.937072     1.324929   
9    7             14  1080        -0.028023      46.005049     1.259501   
10   9             12  1080        -0.028021      45.879462     1.247512   
11  11             10  1080        -0.028011      45.800189     1.246372   
12   5      

In [6]:
db_path = Path("../backtest_results.db")
conn = sqlite3.connect(db_path)

session_comparison = pd.read_sql_query(
    """
    SELECT session,
           COUNT(*) AS n,
           AVG(expectancy) AS mean_expectancy,
           AVG(win_rate) AS mean_win_rate,
           AVG(max_dd) AS mean_max_dd,
           AVG(sharpe_ratio) AS mean_sharpe,
           AVG(profit_factor) AS mean_profit_factor,
           AVG(num_trades) AS mean_num_trades,
           AVG(return_pct) AS mean_return_pct
    FROM """ + TABLE_NAME + """
    WHERE session IS NOT NULL AND sharpe_ratio IS NOT NULL AND expectancy >= 0
    GROUP BY session
    ORDER BY mean_expectancy, mean_win_rate
    """,
    conn,
)
print(session_comparison)

  session   n  mean_expectancy  mean_win_rate  mean_max_dd  mean_sharpe  \
0  LONDON  19         0.001787      42.158217       0.0344     0.082479   

   mean_profit_factor  mean_num_trades  mean_return_pct  
0            1.010326      1933.105263         0.006855  
